In [1]:
# --- replication package paths (auto-inserted) ---
from pathlib import Path

# Resolve the package root whether run from notebooks/ or the root.
_here = Path.cwd()
ROOT = _here if (_here / "data").exists() else _here.parent

DATA    = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

assert DATA.exists(), f"data folder not found: {DATA}"


In [2]:
"""
Alternative H1a/H3 license classifier -- direct dict lookup, for
cross-validating the keyword-list version.

WHY TWO METHODS: the keyword-list version (PERMISSIVE_KEYWORDS /
RECIPROCAL_KEYWORDS, built by hand-copying license_mapping keys grouped by
COMPATIBILITY_GROUPS membership) and this version (a direct dict lookup at
runtime) are mechanically different ways of reaching the same classification.
The keyword-list version does substring matching, which is faster to run but
depends on the copied lists staying in sync with license_mapping if it is
ever edited. This version calls license_mapping and get_license_group()
directly every time, so it can never drift out of sync -- it's always
whatever DSR_engine.py currently says.

If both methods agree on the full dataset, that's strong evidence the
earlier keyword-list transcription was done correctly and the corrected
H1a/H3 numbers are trustworthy. If they disagree anywhere, the disagreement
itself pinpoints exactly which raw string(s) the keyword-list version
mishandled.

USAGE:
    - from classify_license_dict_lookup import classify_license
    df["license_family"] = df["source_file_license"].apply(classify_license)
    - Cross-validation of the license classifier. Compares exact dict lookup against the keyword-list implementation across all 1,183,182 
    deduplicated relationship rows. Run cells in order. Requires license_analysis_results_processed.csv and DSR_engine.py in the working directory.
    Expected result: 0 disagreements.

Then compare against the keyword-list version's output column
(e.g. license_family_keywords) with a crosstab -- see the __main__ block
below for a ready-made comparison routine.
"""

import pandas as pd

# ---- Load DSR_engine.py's mapping + groups directly (single source of truth) ----
_ns = {}
exec(open(DATA / "DSR_engine.py").read(), _ns)  # adjust path if not run from the same dir
license_mapping = _ns["license_mapping"]
get_license_group = _ns["get_license_group"]


def classify_license(lic):
    """Returns 'Permissive', 'Reciprocal', or 'Unresolved'.

    Mechanism: exact dict lookup (license_mapping.get(raw_string_stripped))
    to get an SPDX id, then get_license_group() to bucket that SPDX id.
    No substring matching anywhere -- this is a strict, order-independent
    exact match, unlike the keyword-list version.

    NOTE: license_mapping keys are case-sensitive as written in
    DSR_engine.py (e.g. "MIT License" vs "mit license" are both present as
    separate keys where observed in real data). We only .strip() whitespace,
    matching what DSR_engine.py's own pipeline does -- we do NOT lowercase,
    since license_mapping relies on exact-case keys for some entries.
    """
    if pd.isna(lic):
        return "Unresolved"
    spdx = license_mapping.get(str(lic).strip(), None)
    if spdx is None:
        return "Unresolved"
    group = get_license_group(spdx)
    if group == "Permissive":
        return "Permissive"
    if group in ("Weak Copyleft", "Strong Copyleft"):
        return "Reciprocal"
    return "Unresolved"  # Custom / Restricted / Proprietary / Unknown / Unlicensed


def compare_classifiers(df, keyword_classify_fn, license_col="source_file_license"):
    """
    Runs both classifiers on df[license_col] and returns:
      - a crosstab of (keyword-list result) x (dict-lookup result)
      - the subset of rows where they disagree, with the raw string and
        both classifications, so you can inspect exactly what diverged.
    """
    df = df.copy()
    df["family_keywords"] = df[license_col].apply(keyword_classify_fn)
    df["family_dict_lookup"] = df[license_col].apply(classify_license)

    crosstab = pd.crosstab(
        df["family_keywords"], df["family_dict_lookup"],
        rownames=["keyword-list"], colnames=["dict-lookup"]
    )

    disagreements = df[df["family_keywords"] != df["family_dict_lookup"]][
        [license_col, "family_keywords", "family_dict_lookup"]
    ].drop_duplicates()

    return crosstab, disagreements


if __name__ == "__main__":
    # Quick sanity check with values known to matter from the earlier fix.
    test_values = [
        "GNU General Public License v2.0",
        "Mozilla Public License 2.0",
        "Apache License 2.0",
        "MIT License",
        "European Union Public License 1.2",  # known gap -- see note below
        "The Unlicense",                       # known gap -- see note below
        "Other",
        None,
    ]
    for v in test_values:
        print(f"{str(v)!r:45} -> {classify_license(v)}")

    print(
        "\nNote: 'European Union Public License 1.2' and 'The Unlicense' "
        "resolve to Unresolved here because their SPDX ids (EUPL-1.2, "
        "Unlicense) are not present in any COMPATIBILITY_GROUPS bucket in "
        "DSR_engine.py -- this is a genuine gap in the underlying mapping, "
        "not a bug in this lookup function. Flagged separately; not fixed "
        "here so this function stays a faithful mirror of DSR_engine.py."
    )

'GNU General Public License v2.0'             -> Reciprocal
'Mozilla Public License 2.0'                  -> Reciprocal
'Apache License 2.0'                          -> Permissive
'MIT License'                                 -> Permissive
'European Union Public License 1.2'           -> Unresolved
'The Unlicense'                               -> Unresolved
'Other'                                       -> Unresolved
'None'                                        -> Unresolved

Note: 'European Union Public License 1.2' and 'The Unlicense' resolve to Unresolved here because their SPDX ids (EUPL-1.2, Unlicense) are not present in any COMPATIBILITY_GROUPS bucket in DSR_engine.py -- this is a genuine gap in the underlying mapping, not a bug in this lookup function. Flagged separately; not fixed here so this function stays a faithful mirror of DSR_engine.py.


In [3]:
df = pd.read_csv(DATA / "license_analysis_results_processed.csv")

In [4]:
df["license_family"] = df["source_file_license"].apply(classify_license)

In [5]:
compare_classifiers(df, classify_license, license_col="source_file_license")

(dict-lookup   Permissive  Reciprocal  Unresolved
 keyword-list                                    
 Permissive        427937           0           0
 Reciprocal             0      202735           0
 Unresolved             0           0      552510,
 Empty DataFrame
 Columns: [source_file_license, family_keywords, family_dict_lookup]
 Index: [])